In [ ]:
import itertools
from pathlib import Path

import numpy as np
import numpy.typing as npt
import seaborn as sns
from ising.model import UpdateMethod

from climate_attitudes.dataset import Dataset
from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import configure_mpl
from ising import Ising

RANDOM_SEED = 202606041925

rng = np.random.default_rng(RANDOM_SEED)


np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model/bootstrapped_fit/")

schema = schema.post_index()

In [ ]:
data = np.load(DATA_PATH / "ising_no_use_covariates_no_structure.npz")

In [ ]:
Js = np.stack([Ising.unpack_params(param_vec, k=0)[1] for param_vec in data["params"]])
J_mean = Js.mean(axis=0)

In [ ]:
J_mean[0, 7]

In [ ]:
sns.displot(Js[:, 0, 7])

In [ ]:
sns.displot(Js[:, 0, 7])

In [ ]:
policy_to_belief_cc = J_mean[7, 0]
belief_cc_to_policy = J_mean[0, 7]

In [ ]:
print(
    f"Effect size -- policy support -> belief in climate change: "
    f"{policy_to_belief_cc:.2f}"
)
print(
    f"Effect size -- belief in climate change -> policy support: "
    f"{belief_cc_to_policy:.2f}"
)

Why does this occur?

In [ ]:
datasets = data["Y"]
Y = datasets[0]
Y_t1 = Y[:, 0, :]
Y_t2 = Y[:, 1, :]

support_at_t1 = Y_t1[:, 7] == 1
oppose_at_t1 = Y_t1[:, 7] == -1
support_at_t2 = Y_t2[:, 7] == 1
oppose_at_t2 = Y_t2[:, 7] == -1
believe_at_t1 = Y_t1[:, 0] == 1
skeptic_at_t1 = Y_t1[:, 0] == -1
believe_at_t2 = Y_t2[:, 0] == 1
skeptic_at_t2 = Y_t2[:, 0] == -1

How likely is an individual to retain their belief/skepticism across timesteps?

In [ ]:
p_sustained_belief = (believe_at_t1 & believe_at_t2).sum() / believe_at_t1.sum()
p_sustained_skeptic = (skeptic_at_t1 & skeptic_at_t2).sum() / skeptic_at_t1.sum()

print(f"P(Believe | Prev believe) = {p_sustained_belief:.2f}")
print(f"P(Skeptic | Prev skeptic) = {p_sustained_skeptic:.2f}")

How likely is it that an individual believes in climate change at $t=2$, given that they support climate policy at $t=1$? Or vice versa?

In [ ]:
p_believe_given_support = (support_at_t1 & believe_at_t2).sum() / support_at_t1.sum()
p_skeptic_given_oppose = (oppose_at_t1 & skeptic_at_t2).sum() / oppose_at_t1.sum()

p_support_given_believe = (support_at_t2 & believe_at_t1).sum() / believe_at_t1.sum()
p_oppose_given_skeptic = (oppose_at_t2 & skeptic_at_t1).sum() / skeptic_at_t1.sum()

print(f"P(Believe | Prev support) = {p_believe_given_support:.2f}")
print(f"P(Skeptic | Prev oppose) = {p_skeptic_given_oppose:.2f}")

print()

print(f"P(Support | Prev believe) = {p_support_given_believe:.2f}")
print(f"P(Oppose | Prev skeptic) = {p_oppose_given_skeptic:.2f}")

How likely is it that an individual:
- Supports climate policy, while not believing in climate change, or
- Opposes climate policy, while believing in climate change?

In [ ]:
p_support_and_skeptic = (support_at_t1 & skeptic_at_t1).sum() / support_at_t1.size
p_oppose_and_believe = (oppose_at_t1 & believe_at_t1).sum() / support_at_t1.size

print(f"P(Support and skepticism) = {p_support_and_skeptic:.2f}")
print(f"P(Oppose and believe) = {p_oppose_and_believe:.2f}")

How likely is it that an individual supports climate policy _before_ they believe in climate change? Or opposes it _before_ dropping their belief?

In [ ]:
p_support_before_belief = (
    support_at_t1 & skeptic_at_t1 & believe_at_t2
).sum() / support_at_t1.size
p_oppose_before_skepticism = (
    oppose_at_t1 & believe_at_t1 & skeptic_at_t2
).sum() / oppose_at_t1.size

print(f"P(Belief | Prev support & skeptic) = {p_support_before_belief:.2f}")
print(f"P(Skeptic | Prev oppose & believe) = {p_oppose_before_skepticism:.2f}")

How likely is it that an individual believes in CC _before_ supporting policy? Or is skeptic _before_ opposing it?

In [ ]:
p_believe_before_support = (
    believe_at_t1 & oppose_at_t1 & support_at_t2
).sum() / oppose_at_t1.size
p_skeptic_before_oppose = (
    skeptic_at_t1 & support_at_t1 & oppose_at_t2
).sum() / support_at_t1.size

print(f"P(Support | Prev believe & oppose) = {p_believe_before_support:.2f}")
print(f"P(Oppose | Prev skeptic & support) = {p_skeptic_before_oppose:.2f}")

Probability of belief at $t=2$, given support at $t=1$, conditional on belief or skepticism at $t=1$.

In [ ]:
p_belief_given_support_and_belief = (
    believe_at_t2 & support_at_t1 & believe_at_t1
).sum() / (support_at_t1 & believe_at_t1).sum()
p_belief_given_support_and_skeptic = (
    believe_at_t2 & support_at_t1 & skeptic_at_t1
).sum() / (support_at_t1 & skeptic_at_t1).sum()

p_skeptic_given_oppose_and_belief = (
    skeptic_at_t2 & oppose_at_t1 & believe_at_t1
).sum() / (oppose_at_t1 & believe_at_t1).sum()
p_skeptic_given_oppose_and_skeptic = (
    skeptic_at_t2 & oppose_at_t1 & skeptic_at_t1
).sum() / (oppose_at_t1 & skeptic_at_t1).sum()

print(f"P(Belief | Prev support and belief) = {p_belief_given_support_and_belief:.2f}")
print(
    f"P(Belief | Prev support and skeptic) = {p_belief_given_support_and_skeptic:.2f}"
)
print(f"P(Skeptic | Prev oppose and belief) = {p_skeptic_given_oppose_and_belief:.2f}")
print(
    f"P(Skeptic | Prev oppose and skeptic) = {p_skeptic_given_oppose_and_skeptic:.2f}"
)

Check lagged pair correlations: policy -> belief and belief -> policy. Do these reflect the difference in interaction effects? Are they more interpretable?

## Conditional probabilities

In [ ]:
def pcond(
    x: npt.NDArray[np.int64], y: npt.NDArray[np.int64]
) -> tuple[npt.NDArray[np.float64], npt.NDArray[np.int64], npt.NDArray[np.int64]]:
    """Calculates the conditional probability from two observation vectors.

    For arguments `x` and `y`, estimates the conditional probability P(Y=y|X=x),
    returning a 2D Numpy array in which element (i,j) contains the estimated
    probability P(Y=y_j | X=x_i) = P(Y=y_j, X=x_i)/P(X=x_i), for sorted unique
    values $x_i$, $y_j$.

    Returns:
        Tuple containing probability matrix, v
        ector of sorted unique x values, and
        vector of sorted unique y values.
    """
    x_unique = np.sort(np.unique(x))
    y_unique = np.sort(np.unique(y))
    p = np.zeros((x_unique.size, y_unique.size), dtype=np.float64)

    for i, j in itertools.product(np.arange(x_unique.size), np.arange(y_unique.size)):
        p[i, j] = ((y == y_unique[j]) & (x == x_unique[i])).sum() / (
            x == x_unique[i]
        ).sum()

    return p, x_unique, y_unique

$$P(\text{Belief in CC} \mid \text{Support for climate policy})$$

In [ ]:
P, xvals, yvals = pcond(Y_t1[:, 7], Y_t2[:, 0])
P

$$P(\text{Support for climate policy} \mid \text{Belief in CC})$$

In [ ]:
P, xvals, yvals = pcond(Y_t2[:, 0], Y_t1[:, 7])
P

$$P(\text{Support for climate policy} \mid \text{Support for climate policy})$$

In [ ]:
P, xvals, yvals = pcond(Y_t2[:, 7], Y_t1[:, 7])
P

# Fit model with two spins

Focus only on these spins. If we fit a model to this reduced subset of data, what do we observe in the inferred effects?

We'll look at two cases:

- Model fit with self-interactions
- Model fit without self-interaction

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)
_, Y, _ = dataset.indices_to_numpy(kind="time-series", binarise=True, seed=RANDOM_SEED)
Y = Y[..., [0, 7]]

In [ ]:
# prev_skeptic = Y[:, 0, 0] == -1
# curr_believer = Y[:, 1, 0] == 1
# n_cases = (prev_skeptic & curr_believer).sum()
# remove_idxes = rng.choice(
#     np.arange(n_cases),
#     size=np.round(0.76 * n_cases).astype(np.int64),
#     replace=False,
# )
# to_keep = np.full(Y.shape[0], fill_value=True)
# to_keep[np.argwhere(prev_skeptic & curr_believer)[remove_idxes]] = False
# Y = Y[to_keep]

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 0], Y[:, 1, 0])
P

In [ ]:
bootstrap_models_no_self_loops = []
bootstrap_models = []
for r in range(100):
    idxes = rng.choice(np.arange(Y.shape[0]), size=Y.shape[0], replace=True)
    model_no_self_loops = Ising.fit(
        Y[idxes], update_method=UpdateMethod.SYNCHRONOUS, rng=RANDOM_SEED + r
    )
    model = Ising.fit(
        Y[idxes],
        update_method=UpdateMethod.SYNCHRONOUS,
        rng=RANDOM_SEED + r,
        self_loops=True,
        w=1e-2,
    )
    bootstrap_models_no_self_loops.append(model_no_self_loops)
    bootstrap_models.append(model)

In [ ]:
js_no_self_loops = np.asarray([m.j for m in bootstrap_models_no_self_loops])
js = np.asarray([m.j for m in bootstrap_models])

In [ ]:
sns.displot(np.stack((js_no_self_loops[:, 0, 1], js[:, 0, 1])).T, kind="kde")

In [ ]:
sns.displot(np.stack((js_no_self_loops[:, 0, 1], js[:, 0, 1])).T, kind="kde")

In [ ]:
sns.displot(np.stack((js_no_self_loops[:, 0, 1], js[:, 0, 1])).T, kind="kde")

Check difference between the inferred effects

In [ ]:
no_self_loop_diff = model_no_self_loops.j[0, 1] - model_no_self_loops.j[1, 0]
self_loop_diff = model.j[0, 1] - model.j[1, 0]

print(f"With self-loops: {self_loop_diff:.2f}")
print(f"Without self-loops: {no_self_loop_diff:.2f}")

So introducing self-loops appears to cause (at least) part of this effect. Why?

**First:** Why are the effects approximately equal when we don't consider self-loops?

In this scenario (without self-loops), the state of each spin is determined entirely by the previous state of the other spin.

$$P(\text{Belief in CC} \mid \text{Support for climate policy})$$

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 1], Y[:, 1, 0])
P

When an individual supports climate policy, they almost certainly believe in climate change at the following observation time, irrespective of their previous belief.

However, if an individual _does not_ support climate policy, this constrains their subsequent belief in climate change significantly less. This may be explained by the observation that an individual may oppose climate policy because they do not believe in climate change, but also for a variety of other reasons. On the other hand, if an individual supports climate policy, this is a strong indicator that they believe in climate change.

$$P(\text{Support for climate policy} \mid \text{Belief in CC})$$

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 0], Y[:, 1, 1])
P

We see an analogous trend here. Climate change skeptics almost always subsequently oppose climate policy, while CC believers are relatively more split. In this case, however, the split between support and opposition is not as even as before. Those who believe in climate change are more likely to suppport than to oppose climate policy. On the other hand, there is slightly more entropy in the skepticism case.

We can take a more extended view, conditioning on both prior support and belief.

$$P(\text{Belief in CC} \mid \text{Prior support for climate policy and climate belief})$$

$\text{Climate belief} = \text{belief}$

In [ ]:
Y_belief = Y[Y[:, 0, 0] == 1]
P, xvals, yvals = pcond(Y_belief[:, 0, 1], Y_belief[:, 1, 0])
P

$\text{Climate belief} = \text{skepticism}$

In [ ]:
Y_skepticism = Y[Y[:, 0, 0] == -1]
P, xvals, yvals = pcond(Y_skepticism[:, 0, 1], Y_skepticism[:, 1, 0])
P

$$P(\text{Support for climate policy} \mid \text{Prior support for climate policy and climate belief})$$

$\text{Climate belief} = \text{belief}$

In [ ]:
P, xvals, yvals = pcond(Y_belief[:, 0, 1], Y_belief[:, 1, 1])
P

$\text{Climate belief} = \text{skepticism}$

In [ ]:
P, xvals, yvals = pcond(Y_skepticism[:, 0, 1], Y_skepticism[:, 1, 1])
P

We can calculate the conditional mutual information: "How much do $Y$ and $X_\text{prev}$ say about one another, given that we already know $Y_\text{prev}$?"

In [ ]:
def conditional_entropy(
    x: npt.NDArray[np.int64], y: npt.NDArray[np.int64]
) -> tuple[np.float64, npt.NDArray[np.int64], npt.NDArray[np.int64]]:
    x_unique = np.sort(np.unique(x))
    y_unique = np.sort(np.unique(y))

    # Calculate joint probability table
    p = np.zeros((x_unique.size, y_unique.size), dtype=np.float64)

    for i, j in itertools.product(np.arange(x_unique.size), np.arange(y_unique.size)):
        p[i, j] = ((y == y_unique[j]) & (x == x_unique[i])).sum() / x.size

    # Calculate joint entropy
    H_YX = np.float64(0.0)
    for i, j in itertools.product(np.arange(x_unique.size), np.arange(y_unique.size)):
        H_YX -= p[i, j] * np.log2(p[i, j])

    # Calculate marginal entropy
    H_X = np.float64(0.0)
    for i in np.arange(x_unique.size):
        p_i = p[i].sum()
        H_X -= p_i * np.log2(p_i)

    return H_YX - H_X, x_unique, y_unique

In [ ]:
def conditional_mi(
    y: npt.NDArray[np.int64],
    y_prev: npt.NDArray[np.int64],
    x_prev: npt.NDArray[np.int64],
) -> tuple[npt.NDArray[np.float64], npt.NDArray[np.int64], npt.NDArray[np.int64]]:
    x_prev_unique = np.sort(np.unique(x_prev))
    y_prev_unique = np.sort(np.unique(y_prev))
    y_unique = np.sort(np.unique(y))

    # Calculate H(Y|Y_prev)
    H_Y_given_Yprev = conditional_entropy(y, y_prev)[0]

    # Calculate P(Y, Y_prev, X_prev)
    p = np.zeros(
        (x_prev_unique.size, y_prev_unique.size, y_unique.size), dtype=np.float64
    )
    for i, j, k in itertools.product(
        np.arange(x_prev_unique.size),
        np.arange(y_prev_unique.size),
        np.arange(y_unique.size),
    ):
        p[i, j, k] = (
            (x_prev == x_prev_unique[i])
            & (y_prev == y_prev_unique[j])
            & (y == y_unique[k])
        ).sum() / y.size

    # Calculate H(Y, Y_prev, X_prev)
    H_triple_joint = np.float64(0.0)
    for i, j, k in itertools.product(
        np.arange(x_prev_unique.size),
        np.arange(y_prev_unique.size),
        np.arange(y_unique.size),
    ):
        H_triple_joint -= p[i, j, k] * np.log2(p[i, j, k])

    # Calculate H(Y_prev, X_prev)
    H_pair_joint = np.float64(0.0)
    for i, j in itertools.product(
        np.arange(x_prev_unique.size), np.arange(y_prev_unique.size)
    ):
        H_pair_joint -= p[i, j].sum() * np.log2(p[i, j].sum())

    # Calculate H(Y | X_prev, Y_prev)
    H_Y_given_others = H_triple_joint - H_pair_joint

    # Calculate I(Y; X_prev | Y_prev) = H(Y | Y_prev) - H(Y | X_prev, Y_prev)
    mi = H_Y_given_Yprev - H_Y_given_others

    print(H_Y_given_Yprev)

    return mi, x_prev_unique, y_prev_unique, y_unique

In [ ]:
H, *_ = conditional_mi(Y[:, 1, 0], Y[:, 0, 0], Y[:, 0, 1])
H

In [ ]:
H, *_ = conditional_mi(Y[:, 1, 1], Y[:, 0, 1], Y[:, 0, 0])
H

In [ ]:
model_no_self_loops.j

In [ ]:
H, *_ = conditional_entropy(Y[:, 0, 0], Y[:, 1, 0])
H

In [ ]:
H, *_ = conditional_entropy(Y[:, 0, 1], Y[:, 1, 1])
H

**Autocorrelations**

Belief --> Belief

In [ ]:
np.corrcoef(Y[:, 0, 0], Y[:, 1, 0])

Policy support --> policy support

In [ ]:
np.corrcoef(Y[:, 0, 1], Y[:, 1, 1])

**Cross-lagged correlations**

Policy support --> Belief

In [ ]:
np.corrcoef(Y[:, 0, 1], Y[:, 1, 0])

Belief --> Policy support

In [ ]:
np.corrcoef(Y[:, 0, 0], Y[:, 1, 1])

In [ ]:
def pcorr(x, y, z):
    bx = np.linalg.lstsq(np.column_stack([np.ones(len(z)), z]), x)[0]
    by = np.linalg.lstsq(np.column_stack([np.ones(len(z)), z]), y)[0]

    res_x = x - (bx[0] + bx[1] * z)
    res_y = y - (by[0] + by[1] * z)

    return np.corrcoef(res_x, res_y)[0, 1]

In [ ]:
pcorr(Y[:, 1, 0], Y[:, 0, 1], Y[:, 0, 0])

In [ ]:
pcorr(Y[:, 1, 1], Y[:, 0, 0], Y[:, 0, 1])

In [ ]:
theta_s1 = model_no_self_loops.h[0] + model_no_self_loops.j[1, 0]
theta_s2 = model_no_self_loops.h[0] - model_no_self_loops.j[1, 0]
p_11 = np.exp(theta_s1) / (2 * np.cosh(theta_s1))
p_12 = np.exp(theta_s2) / (2 * np.cosh(theta_s2))

print(f"P(Belief | Support) = {p_11:.3f}")
print(f"P(Belief | Oppose) = {p_12:.3f}")

**Second:** We look at why things change in the self-loops case.

In [ ]:
model.j

The self-interaction on climate policy is larger than on climate belief. This is somewhat surprising, but if correct, may explain the difference in effects. 

Consider an extreme case in which the inferred self-interaction effects are instead $0$ and $1000$ for climate belief and climate policy respectively. This is to say that the state of an individual's belief in climate change is not at all influenced by their previous state, while the state of their support for climate policy is strongly influenced by their previous support or opposition. Given the magnitude of the effects, it is appropriate to say that the individuals current support or opposition to climate policy is **explained by** their previous support or opposition. 

In this scenario, the effect of the individual's belief in climate change on their support is negligible, since the self-interaction provides all necessary explanation. On the other hand, the self-interaction on climate belief explains nothing, so we may expect a larger directional effect from climate policy.

Of course, the real scenario is far subtler than this, but the same principle applies. If the behaviour of a particular belief or attitude is well-explained by a subset of effects (possibly including self-interactions), then effects from spins outside that set are expected to be small. The crucial observation here is that different beliefs and attitudes have _different_ neighborhoods and _different_ incoming interaction effects, so may not be equally-well-explained in absence of directional interactions between them.

**With this in mind** let us now look at whether this scenario (higher consistency on policy stance than belief) is correct/expected. 

$$P(\text{Belief in climate change} \mid \text{Belief in climate change})$$

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 0], Y[:, 1, 0])
P

$$P(\text{Support for climate policy} \mid \text{Support for climate policy})$$

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 1], Y[:, 1, 1])
P

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 0], Y[:, 1, 1])
P

If someone changes from skeptic to believer, how likely are they to switch policy views?

In [ ]:
prev_skeptic = Y[:, 0, 0] == -1
prev_believer = Y[:, 0, 0] == 1
now_skeptic = Y[:, 1, 0] == -1
now_believer = Y[:, 1, 0] == 1
prev_oppose = Y[:, 0, 1] == -1
prev_support = Y[:, 0, 1] == 1
now_oppose = Y[:, 1, 1] == -1
now_support = Y[:, 1, 1] == 1

In [ ]:
((prev_skeptic & now_believer) & (prev_oppose & now_support)).sum() / (
    prev_skeptic & now_believer
).sum()

In [ ]:
((prev_believer & now_skeptic) & (prev_support & now_oppose)).sum() / (
    prev_believer & now_skeptic
).sum()

In [ ]:
((prev_oppose & now_support) & (prev_skeptic & now_believer)).sum() / (
    prev_oppose & now_support
).sum()

In [ ]:
((prev_support & now_oppose) & (prev_skeptic & now_believer)).sum() / (
    prev_support & now_oppose
).sum()

In [ ]:
0.85 * np.log2(1 / 0.85) + 0.15 * np.log2(1 / 0.15)

Indeed we see that policy support or opposition is retained with a high probability, while belief in climate change (or, more accurately, skepticism about climate change) is subject to more variation.

Let's consider why this might be the case. For **belief in climate change**, believers are relatively unlikely to become skeptics, but skeptics are somewhat likely to become believers. This is somewhat expected. However, it's worth recalling that this variable is three-valued (-1, 0, 1), so the binarisation process will result in individuals who respond 'maybe' being pushed to one extreme or the other. This may inflate some of the movement in both directions. 

On the other hand, for **policy support or opposition**, note the following:

- There are relatively few individuals recorded as not believing in climate change
- Individuals who believe in climate change may oppose climate policy for other reasons.

Taken together, we may assume that on average, individuals' support or opposition toward climate policy is driven by factors other than their belief or skepticism about climate change. If these factors are relatively stable over time (e.g., financial position, beliefs about the local impacts of climate change), then we may not expect much change in this variable. 

----
Let's go a bit deeper into this. We see that _belief_ in climate change is quite stable, yet _skepticism_ is not so stable. If skepticism were as stable as belief, then we'd expect a weaker relation from policy support (since the belief would then explain itself well). 

So what happens when we focus on only the skeptics? Let's look at those individuals who do not believe in climate change at $t=1$, and consider the suitability of self-interaction and policy support as predictors of the next state. 

_We could perhaps consider the entropy here. e.g. "how much do we learn about the next state of CC belief given that we add information about Z?"_

In [ ]:
skeptics = Y[:, 0, 0] == -1
Y_skeptic = Y[skeptics]
Y_skeptic

$$H(\text{Belief in climate change} \mid \text{Support for climate policy})$$

In [ ]:
H, *_ = conditional_entropy(Y_skeptic[:, 0, 1], Y_skeptic[:, 1, 0])

In [ ]:
H

$$H(\text{Belief in climate change} \mid \text{Prev belief} = \text{skeptic})$$

In [ ]:
p_change = (Y_skeptic[:, 1, 0] == 1).sum() / Y_skeptic.shape[0]
print(p_change * np.log2(1 / p_change) + (1 - p_change) * np.log2(1 / (1 - p_change)))

In [ ]:
p_change

In [ ]:
print(f"No loops: {model_no_self_loops.h}")
print(f"Loops:    {model.h}")

In [ ]:
model_no_self_loops.j

In [ ]:
model.j

In [ ]:
0.8722 + 0.75322

In [ ]:
0.5956 + 0.550 + 0.583

----

As a last check, let's look at the effects of the binarisation process on these results

In [ ]:
Y = (
    dataset.indices.collect()
    .sort(by=("participant_id", "wave"))
    .select("cc1", "climate_policy")
    .to_numpy()
    .ravel()
    .reshape((-1, 2, 2))
)

In [ ]:
Y[:, :, 1] = Y[:, :, 1] / Y[:, :, 1].std()

In [ ]:
ζ = rng.normal(scale=0.01, size=(Y.shape[0], 2))
Y_hat = Y
Y_hat[:, :, 1] += ζ

very_neg = Y_hat < -0.15
very_pos = Y_hat > 0.15
Y_hat[very_neg] = -1
Y_hat[very_pos] = 1
Y_hat[~very_neg & ~very_pos] = 0
Y = Y_hat.astype(np.int64)

In [ ]:
H, *_ = conditional_entropy(Y[:, 0, 0], Y[:, 1, 0])
H

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 0], Y[:, 1, 0])
P

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 1], Y[:, 1, 1])
P

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 1], Y[:, 1, 0])
P

In [ ]:
P, xvals, yvals = pcond(Y[:, 0, 0], Y[:, 1, 1])
P